In [1]:
import numpy as np
import transforms3d as td
import pandas as pd
import bokeh.io
import bokeh.plotting
# Import libraries
from mpl_toolkits import mplot3d
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.signal import peak_widths, find_peaks, peak_prominences, medfilt
import datetime
from sklearn.decomposition import PCA
import re
import panel as pn
from bokeh.models import ColumnDataSource
from bokeh.palettes import Viridis
bokeh.io.output_notebook()
pn.extension()
import plotly.graph_objects as go
from bokeh.models import HoverTool 
from bokeh.io import export_png

Loading BokehJS ...

In [28]:
class CSVParser(object):
    def __init__(self):
        self.conditions = []
        self.calibration_conditions = []
        self.df = None
        self.calibration_df = None
        self.calibration_dict = None 
        self.angle_plots = {}
        
    def file_to_df(self, filename):
        df = pd.read_csv(filename, low_memory=False)
        df.drop(columns = ['Unnamed: 0'], inplace = True)
        df.sort_values(by=['time'], inplace = True, ignore_index = True)
        df['trakstar0'] = df['trakstar0'].apply(self.parse_transform)
        df['trakstar1'] = df['trakstar1'].apply(self.parse_transform)
        df['trakstar2'] = df['trakstar2'].apply(self.parse_transform)
        df['trakstar_01'] = [np.dot(np.linalg.inv(t1), t2) for t1, t2 in zip(df['trakstar0'], df['trakstar1'])]
        df['trakstar_02'] = [np.dot(np.linalg.inv(t1), t2) for t1, t2 in zip(df['trakstar0'], df['trakstar2'])]
        df["mcp_angle"] = np.empty(len(df.index))
        df["pip_angle"] = np.empty(len(df.index))
        df["finger_angle"] = np.empty(len(df.index))
        df['motor_position'] = df['motor_position'] * 0.007 # encoder to mm
        self.df = df

    def plot_data_pca_plane(self, calibration):

        sub_df = self.df[self.df['condition'] == calibration].copy()
        centered_points = sub_df[['trakstar_02_x', 'trakstar_02_y', 'trakstar_02_z']] - sub_df[['trakstar_02_x', 'trakstar_02_y', 'trakstar_02_z']].mean()
        normal_vector = self.calibration_df[self.calibration_df['condition'] == calibration]['PCA_axis3'].iloc[0]
        pca_axis1 = self.calibration_df[self.calibration_df['condition'] == calibration]['PCA_axis1'].iloc[0]
        pca_axis2 = self.calibration_df[self.calibration_df['condition'] == calibration]['PCA_axis2'].iloc[0]

        # Create a 3D scatter plot for the centered points
        fig = go.Figure()
        fig.add_trace(go.Scatter3d(x=centered_points['trakstar_02_x'], y=centered_points['trakstar_02_y'], z=centered_points['trakstar_02_z'],
                                mode='markers', name='Centered 3D Points'))

        # Plot the normal vector
        fig.add_trace(go.Scatter3d(x=[0, normal_vector[0]], y=[0, normal_vector[1]], z=[0, normal_vector[2]],
                                mode='lines+markers', name='Normal Vector'))
        # Plot the normal vector
        fig.add_trace(go.Scatter3d(x=[-pca_axis1[0], pca_axis1[0]], y=[-pca_axis1[1], pca_axis1[1]], z=[-pca_axis1[2], pca_axis1[2]],
                                mode='lines+markers', name='PCA axis 1'))
        # Plot the normal vector
        fig.add_trace(go.Scatter3d(x=[-pca_axis2[0], pca_axis2[0]], y=[-pca_axis2[1], pca_axis2[1]], z=[-pca_axis2[2], pca_axis2[2]],
                                mode='lines+markers', name='PCA axis 2'))

        # Plot the best-fitting plane
        #x_plane, y_plane, z_plane = np.meshgrid(np.linspace(-pca_axis1[0], pca_axis1[0], 10),
                                        #np.linspace(-pca_axis2[0], pca_axis2[0], 10))

        #fig.add_trace(go.Surface(x=x_plane, y=y_plane, z=z_plane, opacity=0.5, colorscale='Viridis', showscale=False, name='Best-Fitting Plane'))

        # Define the axis range
        axis_range = [-0.25, 0.25]

        # Update layout with explicit axis range
        fig.update_layout(scene=dict(aspectmode="cube",
                                    xaxis=dict(range=axis_range),
                                    yaxis=dict(range=axis_range),
                                    zaxis=dict(range=axis_range)),
            width = 1000,
            height = 800
        )
        # Show the plot
        fig.show()

    def parse_conditions(self):
        '''
        Get all conditions from dataset.
        '''
        conditions = list(self.df['condition'].unique())
        for condition in conditions:
            if 'calibration' in condition:
                self.calibration_conditions.append(condition)

        self.conditions = set(conditions) - set(self.calibration_conditions)

    def parse_transform(self, transform_str):
        # Split the string using regex
        result = re.split(r'[:\n]', transform_str)

        # Remove empty strings from the result
        result = [item.strip() for item in result if item.strip()]
        x = float(result[2])
        y = float(result[4])
        z = float(result[6])
        rx = float(result[9])
        ry = float(result[11])
        rz = float(result[13])
        rw = float(result[15])
        # Display the result
        return td.affines.compose([x,y,z], td.quaternions.quat2mat([rw, rx, ry, rz]), np.ones(3))

    def relative_transform(self, t1, t2):
        return np.dot(np.linalg.inv(t1), t2)

    def get_calibration_pca(self, calibration):
        '''
        input: dataframe & column for calibration
        '''
        data = self.df[self.df['condition'] == calibration]['trakstar_02']
        n = len(data)
        points = np.zeros((n, 3))

        for i, transform in enumerate(data):
            translation = transform[:3, 3]
            points[i, :3] = translation

        pca = PCA()
        pca.fit(points)

        return pca

    def twist_rotation_about_axis(self, transform, axis):
        '''
        Returns the angle theta in degrees about the twist axis. 
        '''
        # quaternion convention is [w, x, y, z]
        transform_quat = td.quaternions.mat2quat(transform[:3, :3])
        transform_axes = np.array([transform_quat[1], transform_quat[2], transform_quat[3]])
        
        dot_product = np.dot(transform_axes, axis)
        dp_sign = np.sign(dot_product)
        axis_norm = np.linalg.norm(axis)
        projection = (dot_product / axis_norm**2) * axis 
        
        # define twist in quaternion form and normalize
        twist = np.array([transform_quat[0], projection[0], projection[1], projection[2]])
        twist /= td.quaternions.qnorm(twist)
        
        # catching singularities
        threshold = 1e-6
        if td.quaternions.qnorm(twist) < threshold:
            print("Singularity in twist")
            return
        elif td.quaternions.qnorm(transform_quat) < threshold:
            print("Singularity in rotation")
            return
        
        twist = dp_sign * twist
        twist_axis, twist_theta = td.quaternions.quat2axangle(twist)

        if not np.allclose(axis, twist_axis):
            print("Axis of rotation is not the given axis. Something went wrong.")
            print("twist axis: %s"%(twist_axis))
            print("pca axis  : %s"%(axis))
        
        return dp_sign*twist_theta*(180/np.pi)

    def calculate_relative_angle(self, ref_transform, target_transform, axis):
        
        net_transform = np.dot(np.linalg.inv(ref_transform), target_transform)
        target_theta = self.twist_rotation_about_axis(net_transform, axis)
        theta_sign = np.sign(target_theta)
        delta = abs(target_theta)
        theta = min(delta, 360 - delta)

        return theta * theta_sign

    def get_calibration_df(self):
        
        calibration_data = []

        for condition in self.calibration_conditions:
            PCA = self.get_calibration_pca(condition)
            pca_axes = PCA.components_
            pca_variance = PCA.explained_variance_ratio_
            mcp_ref_rotation = self.df[self.df['condition'] == condition]['trakstar_01'].iloc[0][:3, :3]
            pip_ref_rotation = self.df[self.df['condition'] == condition]['trakstar_02'].iloc[0][:3, :3]
            calibration_row = {'condition':condition, 
                            'mcp_ref_rotation': mcp_ref_rotation, 
                            'pip_ref_rotation': pip_ref_rotation, 
                            'PCA_axis1':pca_axes[0], 
                            'PCA_axis2':pca_axes[1], 
                            'PCA_axis3':pca_axes[2], 
                            'variance': pca_variance}

            calibration_data.append(calibration_row)
        
        calibration_df = pd.DataFrame(calibration_data)
        self.calibration_df = calibration_df

    def calculate_joint_angles(self, calibration_condition, condition):
        
        joint_axis = self.calibration_df[self.calibration_df['condition']==calibration_condition]['PCA_axis3'].iloc[0]

        # pick the first transform of the dataset as the neutral pose
        mcp_ref_rotation = self.calibration_df[self.calibration_df['condition']==calibration_condition]['mcp_ref_rotation'].iloc[0]
        pip_ref_rotation = self.calibration_df[self.calibration_df['condition']==calibration_condition]['pip_ref_rotation'].iloc[0]
        # Rotate joint axis into frame
        mcp_axis = np.dot(np.linalg.inv(mcp_ref_rotation), joint_axis)
        pip_axis = np.dot(np.linalg.inv(pip_ref_rotation), joint_axis)
        print("Condition: %s"%condition)
        print("\nVariance ratio: %s"%self.calibration_df[self.calibration_df['condition']==calibration_condition]['variance'].iloc[0])
        print("----------------------------\n")

        # Calculate the relative angle using the pre-defined reference transform and joint axis
        mcp_angle = [self.calculate_relative_angle(mcp_ref_rotation, t[:3, :3], mcp_axis) for t in self.df[self.df['condition'] == condition]['trakstar_01']]
        finger_angle = [self.calculate_relative_angle(pip_ref_rotation, t[:3, :3], pip_axis) for t in self.df[self.df['condition'] == condition]['trakstar_02']]
        # Find PIP angle using the difference 
        pip_angle = np.array(np.array([finger_angle]) - np.array([mcp_angle]))[0]
        indices = list(self.df[self.df['condition'] == condition].index)

        self.df.loc[indices, 'mcp_angle'] = mcp_angle
        self.df.loc[indices, 'pip_angle'] = pip_angle
        self.df.loc[indices, 'finger_angle'] = finger_angle

    def get_calibrated_joint_angles(self):

        for condition in self.conditions: 
            self.calculate_joint_angles(self.calibration_dict[condition], condition)

    def finalize_df(self):
        self.df.drop(columns=["trakstar0", "trakstar1", "trakstar2", "trakstar_01"], inplace = True)
        self.df = self.df[~self.df['futek'].isna()].copy()
        self.df = self.df[~self.df['motor_position'].isna()].copy()
        self.df.sort_values(by='raw_time', inplace=True)
        self.df.drop(columns = ['index'], inplace = True)
        self.df.reset_index(drop=True, inplace = True)
        self.df.reset_index(drop=False, inplace = True)
        self.df.rename(columns= {'level_0' : 'index'}, inplace = True)

        self.df.set_index(['condition', 'index'], inplace = True)
        self.df['c_index'] = np.zeros(len(self.df), dtype=int)
        for c in list(self.df.index.levels[0]):
            self.df.loc[(c, slice(None)), 'c_index'] = np.arange(0, len(self.df.loc[(c, slice(None)), 'c_index']), dtype=int)
        self.df.reset_index(inplace = True, drop = False)
        self.df.set_index(['condition', 'c_index', 'index'], inplace=True)
        self.df.rename(columns = {'level_0' : "time_unitless"}, inplace=True)
        self.df.index

    def get_mcp_pip_plot(self, condition):
        x = 'c_index'
        y = 'mcp_angle'
        z = 'pip_angle'
        df = self.df.loc[(condition, slice(None), slice(None)), :].copy().reset_index()
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=800,
                height=600,
                x_axis_label='time units',
                y_axis_label='angle (degrees)',
                title = condition + ' MCP and PIP angles',
                y_range = [-45, 200],
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
            )

        p.circle(df[x], df[y], legend_label = y, alpha = 0.5)
        p.circle(df[x], df[z], color = 'red', legend_label = z, alpha = 0.5)

        return p

    def get_angle_plots(self):
        for i in self.conditions:
            p = self.get_mcp_pip_plot(i)
            p.title.text_font_size = '20pt'
            p.legend.label_text_font_size = '20pt'
            p.xaxis.axis_label_text_font_size = '20pt'
            p.yaxis.axis_label_text_font_size = '20pt'
            p.yaxis.major_label_text_font_size = '18pt'
            #p.legend.click_policy = 'hide'
            bokeh.io.show(p)
            self.angle_plots[i] = p

    def get_xy_plot(self, x, y):

        title = "%s vs. %s"%(y, x)
        col = bokeh.palettes.d3['Category10'][10]
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=1000,
                height=600,
                x_axis_label=x,
                y_axis_label=y,
                title = title,
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
            )

        for i, c in enumerate(self.conditions):
            df = self.df.loc[(c, slice(None), slice(None)), :].copy().reset_index()
            p.circle(df[x], df[y], color = col[i], legend_label = c)

        p.legend.location='top_left'
        p.legend.click_policy='hide'
        bokeh.io.show(p)
        return p
    
    def save_angle_plots(self):
        for p, name in self.angle_plots.items():
            export_png(p, filename="%s_angle_plot.png"%name)
        
def main(filename):
    csv_file = CSVParser()
    csv_file.file_to_df(filename)
    csv_file.parse_conditions()
    return csv_file



In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/p3_rosbags/p3_compiled_data.csv')
df.reset_index(inplace = True)
df.rename(columns={"Unnamed: 0" : "time_unitless"}, inplace=True)

df = df[~df['futek'].isna()].copy()
df = df[~df['motor_position'].isna()].copy()
df['motor_position'] = 0.007*df['motor_position']
df.sort_values(by='raw_time', inplace=True)
df.drop(columns = ['index'], inplace = True)
df.reset_index(drop=True, inplace = True)
df.reset_index(drop=False, inplace = True)
df.rename(columns= {'level_0' : 'index'}, inplace = True)

df.set_index(['condition', 'index'], inplace = True)
df['c_index'] = np.zeros(len(df), dtype=int)
for c in list(df.index.levels[0]):
    df.loc[(c, slice(None)), 'c_index'] = np.arange(0, len(df.loc[(c, slice(None)), 'c_index']), dtype=int)
df.reset_index(inplace = True, drop = False)
df.set_index(['condition', 'c_index', 'index'], inplace=True)
df.rename(columns = {'level_0' : "time_unitless"}, inplace=True)
df

time_unitless      raw_time  \
condition                 c_index index                                
passive_start_calibration 0       0              44042  1.706732e+09   
                          1       1              44043  1.706732e+09   
                          2       2              44046  1.706732e+09   
                          3       3              44047  1.706732e+09   
                          4       4              44049  1.706732e+09   
...                                                ...           ...   
passive_end               2404    47360          25334  1.706735e+09   
                          2405    47361          25335  1.706735e+09   
                          2406    47362          25338  1.706735e+09   
                          2407    47363          25341  1.706735e+09   
                          2408    47364          25342  1.706735e+09   

                                                           time  futek  \
condition                 c_index index                                  
passive_start_calibration 0       0      2024-01-31 20:13:32.42    0.0   
                          1       1      2024-01-31 20:13:32.44    0.0   
                          2       2      2024-01-31 20:13:32.48    0.0   
                          3       3      2024-01-31 20:13:32.50    0.0   
                          4       4      2024-01-31 20:13:32.53    0.0   
...                                                         ...    ...   
passive_end               2404    47360  2024-01-31 20:57:49.63    0.0   
                          2405    47361  2024-01-31 20:57:49.65    0.0   
                          2406    47362  2024-01-31 20:57:49.69    0.0   
                          2407    47363  2024-01-31 20:57:49.73    0.0   
                          2408    47364  2024-01-31 20:57:49.75    0.0   

                                         motor_position  
condition                 c_index index                  
passive_start_calibration 0       0                0.00  
                          1       1                0.00  
                          2       2                0.00  
                          3       3                0.00  
                          4       4                0.00  
...                                                 ...  
passive_end               2404    47360            0.07  
                          2405    47361            0.07  
                          2406    47362            0.07  
                          2407    47363            0.07  
                          2408    47364            0.07  

[47365 rows x 5 columns]

In [7]:
df.to_csv('p3_parsed_data.csv')

# Import data

In [39]:
#filename = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/demohand2/demohand1_compiled_data.csv'
#filename = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/test1_rosbags/tf_data.csv'
#filename = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/p1_jan_23_2024/p1_compiled_data.csv'
#filename = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/p13_rosbags/p13_compiled_data.csv'
filename = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/p1_jan_23_2024/p1_compiled_data.csv'
data = main(filename) 
print("Conditions: %s\nCalibration conditions: %s"%(data.conditions, data.calibration_conditions))


Conditions: {'passive_start', 'passive_end', 'active_cube', 'active_rorcr'}
Calibration conditions: ['passive_start_calibration', 'active_calibration', 'passive_end_calibration']


In [40]:
calibration_dict = {'passive_start': 'passive_start_calibration', 'active_rorcr': 'active_calibration', 'active_cube' : 'active_calibration', 'passive_end':'passive_end_calibration'}
data.calibration_dict = calibration_dict
data.get_calibration_df()
data.get_calibrated_joint_angles()
data.finalize_df()

Condition: passive_start

Variance ratio: [0.84969457 0.1435024  0.00680304]
----------------------------

Condition: passive_end

Variance ratio: [0.70581129 0.27656261 0.0176261 ]
----------------------------

Condition: active_cube

Variance ratio: [0.901043   0.07068741 0.02826959]
----------------------------

Condition: active_rorcr

Variance ratio: [0.901043   0.07068741 0.02826959]
----------------------------



In [41]:
data.df

raw_time                    time  \
condition                 c_index index                                         
passive_start_calibration 0       0      1.706026e+09  2024-01-23 16:05:56.63   
                          1       1      1.706026e+09  2024-01-23 16:05:56.67   
                          2       2      1.706026e+09  2024-01-23 16:05:56.73   
                          3       3      1.706026e+09  2024-01-23 16:05:56.75   
                          4       4      1.706026e+09  2024-01-23 16:05:56.83   
...                                               ...                     ...   
passive_end               1202    36057  1.706029e+09  2024-01-23 17:02:44.36   
                          1203    36058  1.706029e+09  2024-01-23 17:02:44.40   
                          1204    36059  1.706029e+09  2024-01-23 17:02:44.43   
                          1205    36060  1.706029e+09  2024-01-23 17:02:44.45   
                          1206    36061  1.706029e+09  2024-01-23 17:02:44.47   

                                            futek  motor_position  \
condition                 c_index index                             
passive_start_calibration 0       0      0.000000           0.000   
                          1       1      0.000000           0.000   
                          2       2      0.000000           0.000   
                          3       3      0.000000           0.000   
                          4       4      0.000000           0.000   
...                                           ...             ...   
passive_end               1202    36057  0.000000           0.189   
                          1203    36058  0.000000           0.189   
                          1204    36059  0.000000           0.189   
                          1205    36060  0.287921           0.189   
                          1206    36061  0.000000           0.189   

                                                                               trakstar_02  \
condition                 c_index index                                                      
passive_start_calibration 0       0      [[0.2419841444444088, -0.9517715058239867, -0....   
                          1       1      [[0.2408784899137337, -0.9522991257006145, -0....   
                          2       2      [[0.23855162660777918, -0.9532391591783221, -0...   
                          3       3      [[0.23866698459899732, -0.9532837041109504, -0...   
                          4       4      [[0.2386352762855732, -0.9535152842454473, -0....   
...                                                                                    ...   
passive_end               1202    36057  [[-0.9902230562185056, -0.13471670135194563, -...   
                          1203    36058  [[-0.9903091506312895, -0.13411893653064041, -...   
                          1204    36059  [[-0.9904006854800302, -0.1333536695462803, -0...   
                          1205    36060  [[-0.9904489280700535, -0.13296882817245345, -...   
                          1206    36061  [[-0.9905259594718752, -0.1323743930673121, -0...   

                                             mcp_angle      pip_angle  \
condition                 c_index index                                 
passive_start_calibration 0       0      1.640891e-315   0.000000e+00   
                          1       1      6.907204e-310  6.907204e-310   
                          2       2      6.907204e-310  6.907204e-310   
                          3       3      6.907204e-310  6.907204e-310   
                          4       4      6.907204e-310  6.907204e-310   
...                                                ...            ...   
passive_end               1202    36057   8.442736e+00   6.796614e+01   
                          1203    36058   8.429139e+00   6.801536e+01   
                          1204    36059   8.424636e+00   6.806325e+01   
                          1205    36060   8.436597e+00   6.807338e+

In [340]:
dff = p13_data.df.loc[('passive_start_calibration', slice(None), slice(None)), :].reset_index()
dff.head(3)

,condition,c_index,index,raw_time,time,futek,motor_position,trakstar_02,mcp_angle,pip_angle,finger_angle
0,passive_start_calibration,0,0,1.706714e+09,2024-01-31 15:08:17.72,0.0,0.0,"[[0.23543548380113866, 0.9710664263371758, 0.0...",4.326959e-315,0.000000e+00,0.000000e+00
1,passive_start_calibration,1,1,1.706714e+09,2024-01-31 15:08:17.76,0.0,0.0,"[[0.23805716513238984, 0.9704254183594058, 0.0...",6.925027e-310,6.925027e-310,6.925027e-310
2,passive_start_calibration,2,2,1.706714e+09,2024-01-31 15:08:17.78,0.0,0.0,"[[0.2391370399502256, 0.9701456898201141, 0.04...",6.925027e-310,6.925027e-310,6.925027e-310


# old

In [8]:
df = pd.read_csv('/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/rosbag/p3_rosbags/p3_compiled_data.csv')
df.reset_index(inplace = True)
df.rename(columns={"Unnamed: 0" : "time_unitless"}, inplace=True)

df = df[~df['futek'].isna()].copy()
df = df[~df['motor_position'].isna()].copy()
df.sort_values(by='raw_time', inplace=True)
df.drop(columns = ['index'], inplace = True)
df.reset_index(drop=True, inplace = True)
df.reset_index(drop=False, inplace = True)
df.rename(columns= {'level_0' : 'index'}, inplace = True)

df.set_index(['condition', 'index'], inplace = True)
df['c_index'] = np.zeros(len(df), dtype=int)
for c in list(df.index.levels[0]):
    df.loc[(c, slice(None)), 'c_index'] = np.arange(0, len(df.loc[(c, slice(None)), 'c_index']), dtype=int)
df.reset_index(inplace = True, drop = False)
df.set_index(['condition', 'c_index', 'index'], inplace=True)
df.rename(columns = {'level_0' : "time_unitless"}, inplace=True)
df


time_unitless      raw_time  \
condition                 c_index index                                
passive_start_calibration 0       0              44042  1.706732e+09   
                          1       1              44043  1.706732e+09   
                          2       2              44046  1.706732e+09   
                          3       3              44047  1.706732e+09   
                          4       4              44049  1.706732e+09   
...                                                ...           ...   
passive_end               2404    47360          25334  1.706735e+09   
                          2405    47361          25335  1.706735e+09   
                          2406    47362          25338  1.706735e+09   
                          2407    47363          25341  1.706735e+09   
                          2408    47364          25342  1.706735e+09   

                                                           time  futek  \
condition                 c_index index                                  
passive_start_calibration 0       0      2024-01-31 20:13:32.42    0.0   
                          1       1      2024-01-31 20:13:32.44    0.0   
                          2       2      2024-01-31 20:13:32.48    0.0   
                          3       3      2024-01-31 20:13:32.50    0.0   
                          4       4      2024-01-31 20:13:32.53    0.0   
...                                                         ...    ...   
passive_end               2404    47360  2024-01-31 20:57:49.63    0.0   
                          2405    47361  2024-01-31 20:57:49.65    0.0   
                          2406    47362  2024-01-31 20:57:49.69    0.0   
                          2407    47363  2024-01-31 20:57:49.73    0.0   
                          2408    47364  2024-01-31 20:57:49.75    0.0   

                                         motor_position  
condition                 c_index index                  
passive_start_calibration 0       0                 0.0  
                          1       1                 0.0  
                          2       2                 0.0  
                          3       3                 0.0  
                          4       4                 0.0  
...                                                 ...  
passive_end               2404    47360            10.0  
                          2405    47361            10.0  
                          2406    47362            10.0  
                          2407    47363            10.0  
                          2408    47364            10.0  

[47365 rows x 5 columns]